In [1]:
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
path_to_raw = Path("../results/faiss/raw_search_results.jsonl")
path_to_harmonized = Path("../results/faiss/harmonized_search_results.jsonl")

def read_jsonl(path):
    data = []
    with path.open("r") as f:
        for line in f:
            data.append(json.loads(line))
            
    return data

raw = read_jsonl(path_to_raw)
harm = read_jsonl(path_to_harmonized)

In [16]:
descs = []
for res in harm[0]["results"]:
    d = res["descriptor"]
    descs.append(d)


with open("../results/faiss/all_sarcasm_descriptors_harm.jsonl", "w") as f:
    for d in descs:
        f.write(f"{json.dumps({'descriptor': d})}\n")

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

query = "sarcasm; the document contains sarcastic humor and irony"
descriptors = []
for desc in raw[0]["results"]:
    descriptors.append(desc["descriptor"])

# Rerank documents
results = model.rerank(query, descriptors)

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

config.json:   0%|          | 0.00/828 [00:00<?, ?B/s]

modeling.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v3:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

In [45]:
def get_doc_ids(data, annotation="no"):
    doc_ids = set()
    descs_kept = 0
    descs_rejected = 0
    print("Query:", data[0]["query"])
    for res in data[0]["results"]:
        if annotation=="manual":
            print(res["distance"])
            print(res["descriptor"])
            good_hit = input("Is this a good hit? y/n")
            if good_hit.lower() == "y":
                for doc in res["documents"]:
                    doc_ids.add(doc["doc_id"])
                descs_kept += 1
            else:
                descs_rejected += 1
                continue
        elif annotation=="auto":
            hit_words = ["sarcas", "irony", "ironic"]
            for word in hit_words:
                if word in res["descriptor"]:
                    for doc in res["documents"]:
                        doc_ids.add(doc["doc_id"])
                    keep = True
                    break
                else:
                    keep = False
                    continue
            if keep:
                descs_kept += 1
            else:
                descs_rejected += 1
        else:
            for doc in res["documents"]:
                doc_ids.add(doc["doc_id"])

    
    print("Kept", descs_kept)
    print("Rejected", descs_rejected)
    
    return doc_ids

all_raw_ids = get_doc_ids(raw, annotation="auto")
all_harm_ids = get_doc_ids(harm, annotation="auto")

Query: sarcasm; the document contains sarcastic humor and irony
Kept 4167
Rejected 3661
Query: sarcasm; the document contains sarcastic humor and irony
Kept 128
Rejected 108


In [42]:
len(all_harm_ids)

7398

In [35]:
len(valid_harm_ids)

4106

In [25]:
raw_in_harm = 0
for i in raw_ids:
    if i in harm_ids:
        raw_in_harm += 1

harm_in_raw = 0
for i in harm_ids:
    if i in raw_ids:
        harm_in_raw += 1

print(raw_in_harm)
print(harm_in_raw)

5049
5049


In [37]:
len(valid_harm_ids & valid_raw_ids)

2683

In [6]:
path_to_valid_descs_raw = Path("../results/LLM_as_judge/sarcasm_descriptor_correspondence_raw.jsonl")
path_to_valid_descs_harm = Path("../results/LLM_as_judge/sarcasm_descriptor_correspondence_harmonized.jsonl")

raw_descs_responses = read_jsonl(path_to_valid_descs_raw)
harm_descs_responses = read_jsonl(path_to_valid_descs_harm)

def parse_responses(responses):
    valid_descriptors = []
    for response in responses:
        desc = response["example"]["descriptor"]
        answer = response["response"]
        if answer.lower().strip().endswith("answer: yes"):
            valid_descriptors.append(desc)

    return valid_descriptors

raw_valid_descs = parse_responses(raw_descs_responses)
harm_valid_descs = parse_responses(harm_descs_responses)

In [16]:
def get_valid_docs(res, descs):
    valid_docs = []
    for result in res[0]["results"]:
        if result["descriptor"] in descs:
            for doc in result["documents"]:
                valid_docs.append(doc["doc_id"])

    # remove possible duplicates
    valid_docs = list(set(valid_docs))
    return valid_docs


harm_doc_ids = get_valid_docs(harm, harm_valid_descs)
raw_doc_ids = get_valid_docs(raw, raw_valid_descs)

In [27]:
len(set(raw_doc_ids) & set(harm_doc_ids))

4568

In [21]:
def save_doc_ids(data, dtype):
    with open(f"../results/faiss/valid_sarcasm_docs_{dtype}.txt", "w") as f:
        for line in data:
            f.write(line)
            f.write("\n")

save_doc_ids(harm_doc_ids, "harmonized")
save_doc_ids(raw_doc_ids, "raw")